In [50]:
from halib import *

IN_CSV = r"/mnt/e/SyncData/paper2_main/zout/compare/cmp_results.csv"
df = pd.read_csv(IN_CSV, sep=";", encoding="utf-8")
METHODS = []
METHODS = [col.replace("_correct", "") for col in df.columns if "_correct" in col]
df.head()

,video,gt,total_frames,prof_hgnetv2b5_2classes_notemp_correct,prof_hgnetv2b5_2classes_notemp_num_wrong_frames,yolov5s_notemp_correct,yolov5s_notemp_num_wrong_frames,yolov5l_notemp_correct,yolov5l_notemp_num_wrong_frames
0,FP1,X_None,30,True,0,True,0,True,0
1,FP11,X_None,60,True,0,False,1,False,2
2,FP12,X_None,60,True,0,True,0,True,0
3,FP13,X_None,60,True,0,False,13,False,7
4,FP14,X_None,3,True,0,True,0,True,0


In [51]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

POS = "O_Fire_smoke"
NEG = "X_None"

def invert_label(label):
    if label == POS:
        return NEG
    elif label == NEG:
        return POS
    else:
        raise ValueError(f"Unknown label: {label}")

def get_gt_and_pred(df, method, mode='per_video'): # mode: per_frame or per_video
    assert mode in ['per_frame', 'per_video'], "mode should be 'per_frame' or 'per_video'"

    if mode == 'per_video':
        y_true = df["gt"].tolist()
        y_correct = df[f"{method}_correct"].tolist()
        y_pred = []
        y_correct = df[f"{method}_correct"].tolist()
        y_true = df["gt"].tolist()
        y_pred = []
        for i in range(len(y_correct)):
            if y_correct[i] is True: # predicted correctly
                y_pred.append(y_true[i])
            else: # predicted incorrectly
                y_pred.append(invert_label(y_true[i]))
    elif mode == 'per_frame':
        y_true_video = df["gt"].tolist()
        num_videos = len(y_true_video)
        frm_cnt_video = df["total_frames"].tolist()
        y_true = []
        num_wrongs_frames_video = df[f"{method}_num_wrong_frames"].tolist()
        y_pred = []
        for i in range(num_videos):
            num_total = frm_cnt_video[i]
            y_true.extend([y_true_video[i]] * num_total)
            num_wrong_frames = num_wrongs_frames_video[i]
            num_correct_frames = num_total - num_wrong_frames
            y_pred.extend([y_true_video[i]] * num_correct_frames)
            y_pred.extend([invert_label(y_true_video[i])] * num_wrong_frames)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return y_true, y_pred


def cal_metric(df, methods, mode='per_video'): # mode: per_frame or per_video
    assert mode in ['per_frame', 'per_video'], "mode should be 'per_frame' or 'per_video'"
    results = []
    for method in methods:
        method_rs_dict = {'method': method}
        y_true, y_pred = get_gt_and_pred(df, method, mode=mode)
        try:
            # Calculate metrics
            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred, pos_label=POS)
            recall = recall_score(y_true, y_pred, pos_label=POS)
            f1 = f1_score(y_true, y_pred, pos_label=POS)

            # Compute confusion matrix (labels ordered as [negative, positive])
            tn, fp, fn, tp = confusion_matrix(
                y_true, y_pred, labels=[NEG, POS]
            ).ravel()
            false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
            # save results as a dict
            method_rs_dict.update(
                {
                    "accuracy": accuracy * 100.0,
                    "f1_score": f1 * 100.0,
                    "recall": recall * 100.0,
                    "FAR (false_alarm_rate)": false_alarm_rate * 100.0,
                    "precision": precision * 100.0,
                }
            )
            results.append(method_rs_dict)

        except ValueError as e:
            print(f"Error: {e}. Please ensure labels are valid strings and lists have the same length.")
    # Convert results to DataFrame for better visualization
    # pprint(results)
    results_df = pd.DataFrame(results)
    return results_df
# df = df.head(2) # for testing
# filter df by video col if video in ['FP13', 'FP39', 'VP16', 'VP8']
# pprint(len(df))
proc_df = df.copy()
# proc_df = df[df['video'].isin(['FP13', 'FP39', 'VP16', 'VP8'])]
# csvfile.show(proc_df)

# proc_df.to_csv("./check_filtered.csv", index=False, sep=";", encoding="utf-8")

per_video_df = cal_metric(proc_df, METHODS, mode='per_video')
per_frame_df = cal_metric(proc_df, METHODS, mode='per_frame')

console.rule("[bold red]Per Video Results")
csvfile.show(per_video_df)
per_video_df.to_csv("./zout/compare/per_video_results.csv", index=False, sep=";", encoding="utf-8")
console.rule("[bold red]Per Frame Results")
csvfile.show(per_frame_df)
per_frame_df.to_csv("./zout/compare/per_frame_results.csv", index=False, sep=";", encoding="utf-8")

──────────────────────────────────────────────── Per Video Results ────────────────────────────────────────────────

Loading ITables v2.4.4 from the internet... (need help?)


──────────────────────────────────────────────── Per Frame Results ────────────────────────────────────────────────

Loading ITables v2.4.4 from the internet... (need help?)


In [6]:

# -----------------------------
# FFmpeg Horizontal Stack
# -----------------------------
def video_hstack(video_files, output_file):
    """Horizontally stack multiple videos using FFmpeg."""
    tmp_file = "./video_list.txt"
    try:
        with open(tmp_file, "w") as f:
            for video in video_files:
                pprint(video)
                f.write(f"file '{video}'\n")

        ffmpeg_cmd = (
            f"ffmpeg -f concat -safe 0 -i {tmp_file} "
            f'-filter_complex "[0:v][1:v][2:v]hstack=inputs={len(video_files)}[v]" '
            f'-map "[v]" -c:v libx264 -preset fast -crf 22 {output_file}'
        )
        pprint(ffmpeg_cmd)

        os.system(ffmpeg_cmd)
        print(f"[INFO] Video stacked successfully: {output_file}")

    except Exception as e:
        print(f"[ERROR] Video stacking failed: {e}")
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

In [7]:
from halib import *
wrong_df = pd.read_csv("./zout/compare/method_wrong.csv", sep=";", encoding="utf-8")
methods = [col.replace("_num_wrong_frames", "") for col in wrong_df.columns if "_num_wrong_frames" in col]
pprint(methods)

vis_df = pd.read_csv("./zout/compare/cmp_raw_input.csv", sep=";", encoding="utf-8")

vis_df  = vis_df[vis_df['video'].isin(wrong_df['video'].tolist())]
vis_cols = [f'{method}_vis' for method in methods]
vis_proc_df = vis_df[['video'] + vis_cols]
csvfile.show(vis_proc_df)


VIDEODIR = r"/mnt/e/SyncData/paper2_main/zout/compare/vis"
for idx, row in vis_proc_df.iterrows():
    video_files = [os.path.join(VIDEODIR, row[f"{method}_vis"]) for method in methods]
    output_file = os.path.join(VIDEODIR, f"{row['video']}_hstack.mp4")
    pprint(video_files)
    video_hstack(video_files, output_file)
    break
    # video_hstack(video_files, output_file)

['prof_hgnetv2b5_2classes_notemp', 'yolov5s_notemp', 'yolov5l_notemp']

Loading ITables v2.4.4 from the internet... (need help?)


[
│   '/mnt/e/SyncData/paper2_baseline/zout/DFire_test/prof_hgnetv2b5_2classes_notemp/FP11_out.mp4',
│   '/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5s_notemp/vis/FP11_vis.mp4',
│   '/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5l_notemp/vis/FP11_vis.mp4'
]

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/prof_hgnetv2b5_2classes_notemp/FP11_out.mp4'

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5s_notemp/vis/FP11_vis.mp4'

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5l_notemp/vis/FP11_vis.mp4'

'ffmpeg -f concat -safe 0 -i ./video_list.txt -filter_complex "[0:v][1:v][2:v]hstack=inputs=3[v]" -map "[v]" -c:v libx264 -preset fast -crf 22 /mnt/e/SyncData/paper2_main/zout/compare/vis/FP11_hstack.mp4'

[INFO] Video stacked successfully: /mnt/e/SyncData/paper2_main/zout/compare/vis/FP11_hstack.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena